[![Mở bằng Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Puuffiiii/261-ML-Assignment-G3/blob/feature/preprocessing/notebooks/ML_Tabular_Final.ipynb)

# EDA, tiền xử lý và Feature Engineering — HR Analytics

**Mục tiêu:** hợp nhất toàn bộ công việc khám phá dữ liệu, làm sạch, tiền xử lý và tạo đặc trưng thành một quy trình duy nhất, có thể chạy lại từ đầu đến cuối và xuất dữ liệu sẵn sàng cho **Machine Learning** lẫn **Deep Learning**.

Notebook tuân thủ ba nguyên tắc:

1. Chia dữ liệu trước khi học mọi thống kê để tránh rò rỉ dữ liệu.
2. Chỉ `fit` imputation, encoding, scaling và PCA trên tập train.
3. Giữ riêng validation cho chọn mô hình/early stopping và test cho đánh giá cuối cùng.

## 0. Luồng xử lý và khả năng tái lập

`Dữ liệu thô → kiểm tra chất lượng → EDA → split 70/15/15 → imputation → encoding → feature engineering → scaling → PCA → kiểm định → xuất artefact`

- Seed cố định: `42`.
- Đầu ra chính: các mảng `.npy`, tên đặc trưng, metadata, chỉ số split và preprocessor đã fit trong `features/`.
- Các biểu đồ EDA được lưu tại `reports/figures/`.

In [1]:
import os
import sys
import json
import platform
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-ml-assignment")

# Tự nhận diện môi trường Local/Colab.
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/Puuffiiii/261-ML-Assignment-G3.git"
    REPO_NAME = "261-ML-Assignment-G3"
    BRANCH = "feature/preprocessing"
    if not Path(REPO_NAME).exists():
        !git clone -b {BRANCH} {REPO_URL}
    PROJECT_ROOT = Path(REPO_NAME).resolve()
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = current.parent if (current.parent / "modules").exists() else current

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, roc_auc_score
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight

from modules.eda_viz import (
    check_city_cdi_relationship,
    get_missing_stats,
    plot_correlation_matrix,
    plot_high_cardinality_feature,
    plot_missing_values,
    plot_missingness_vs_target,
    plot_nominal_feature,
    plot_numerical_feature,
    plot_ordinal_feature,
    plot_target_distribution,
)
from modules.preprocessing import (
    ENGINEERED_FEATURES,
    NOMINAL_COLS,
    ORDINAL_MAPPINGS,
    TabularPreprocessor,
    load_processed_features,
    load_raw_dataset,
    save_processed_features,
    split_train_validation_test,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
DATA_PATH = PROJECT_ROOT / "data" / "aug_train.csv"
FEATURES_DIR = PROJECT_ROOT / "features"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
print(f"Project root: {PROJECT_ROOT}")
print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

Project root: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3
Python 3.14.7 | pandas 3.0.6 | scikit-learn 1.9.1


## 1. Nạp dữ liệu và kiểm tra chất lượng

Tập dữ liệu gồm hồ sơ ứng viên ngành khoa học dữ liệu. Biến đích `target` nhận `1` nếu ứng viên muốn thay đổi công việc và `0` nếu không.

In [2]:
df_raw = load_raw_dataset(str(DATA_PATH))
print(f"Kích thước dữ liệu: {df_raw.shape[0]:,} dòng × {df_raw.shape[1]} cột")
display(df_raw.head())
display(df_raw.dtypes.rename("dtype").to_frame().T)

Kích thước dữ liệu: 19,158 dòng × 14 cột


,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,8949,city_103,0.920,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,NaN,NaN,1,36,1.0
1,29725,city_40,0.776,Male,No relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,>4,47,0.0
2,11561,city_21,0.624,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,83,0.0
3,33241,city_115,0.789,NaN,No relevent experience,NaN,Graduate,Business Degree,<1,NaN,Pvt Ltd,never,52,1.0
4,666,city_162,0.767,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,50-99,Funded Startup,4,8,0.0


,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
dtype,int64,str,float64,str,str,str,str,str,str,str,str,str,int64,float64


In [3]:
quality_summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values(["missing_pct", "n_unique"], ascending=[False, False])

duplicate_rows = int(df_raw.duplicated().sum())
duplicate_ids = int(df_raw["enrollee_id"].duplicated().sum())
invalid_targets = int((~df_raw["target"].isin([0, 1])).sum())
print({
    "dòng_trùng_hoàn_toàn": duplicate_rows,
    "enrollee_id_trùng": duplicate_ids,
    "target_không_hợp_lệ": invalid_targets,
})
display(quality_summary)

assert duplicate_ids == 0
assert invalid_targets == 0

{'dòng_trùng_hoàn_toàn': 0, 'enrollee_id_trùng': 0, 'target_không_hợp_lệ': 0}


,dtype,missing_count,missing_pct,n_unique
company_type,str,6140,32.05,6
company_size,str,5938,30.99,8
gender,str,4508,23.53,3
major_discipline,str,2813,14.68,6
education_level,str,460,2.40,5
last_new_job,str,423,2.21,6
enrolled_university,str,386,2.01,3
experience,str,65,0.34,22
enrollee_id,int64,0,0.00,19158
training_hours,int64,0,0.00,241


## 2. Exploratory Data Analysis (EDA)

Phần này trả lời bốn câu hỏi: target mất cân bằng đến đâu, biến nào có tín hiệu dự báo, missingness có mang thông tin hay không, và cột nào nên loại bỏ trước mô hình hóa.

In [4]:
target_summary = df_raw["target"].value_counts().sort_index().to_frame("count")
target_summary["pct"] = (target_summary["count"] / len(df_raw) * 100).round(2)
display(target_summary)
plot_target_distribution(
    df_raw,
    save_path=str(FIGURES_DIR / "group1_target_distribution.png"),
    show=True,
)

,count,pct
target,,
0.0,14381,75.07
1.0,4777,24.93


findfont: Failed to find font weight semibold for DejaVu Sans, now using 700.


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group1_target_distribution.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
missing_stats = get_missing_stats(df_raw)
display(missing_stats[missing_stats["Số dòng khuyết"] > 0])
plot_missing_values(
    df_raw,
    save_path=str(FIGURES_DIR / "group6_missing_values_bar.png"),
    show=True,
)

,Kiểu dữ liệu,Số dòng khuyết,Tỷ lệ khuyết (%),Số giá trị duy nhất
company_type,str,6140,32.05,6
company_size,str,5938,30.99,8
gender,str,4508,23.53,3
major_discipline,str,2813,14.68,6
education_level,str,460,2.40,5
last_new_job,str,423,2.21,6
enrolled_university,str,386,2.01,3
experience,str,65,0.34,22


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group6_missing_values_bar.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:444: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
numerical_cols = ["city_development_index", "training_hours"]
display(df_raw.groupby("target")[numerical_cols].agg(["mean", "median", "std"]).round(3))

for col in numerical_cols:
    plot_numerical_feature(
        df_raw,
        col,
        save_path=str(FIGURES_DIR / f"group2_num_{col}.png"),
        show=True,
    )

city_development_index               training_hours               
                         mean median    std           mean median     std
target                                                                   
0.0                     0.853   0.91  0.105         66.114   48.0  60.939
1.0                     0.756   0.74  0.143         63.118   46.0  57.274

-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group2_num_city_development_index.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:163: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group2_num_training_hours.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:163: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# Bảng gọn: tỷ lệ target=1 theo từng nhóm của toàn bộ biến phân loại/thứ bậc.
categorical_cols = [
    "gender", "relevent_experience", "enrolled_university",
    "education_level", "major_discipline", "experience",
    "company_size", "company_type", "last_new_job",
]
category_rows = []
for col in categorical_cols:
    temp = df_raw[[col, "target"]].copy()
    temp[col] = temp[col].fillna("(Missing)")
    stats = temp.groupby(col, observed=False)["target"].agg(["size", "mean"]).reset_index()
    stats["feature"] = col
    stats["target_rate_pct"] = (stats["mean"] * 100).round(2)
    category_rows.append(stats.rename(columns={col: "category"})[[
        "feature", "category", "size", "target_rate_pct"
    ]])
category_target_rates = pd.concat(category_rows, ignore_index=True)
display(category_target_rates.sort_values(["feature", "target_rate_pct"], ascending=[True, False]))
category_target_rates.to_csv(REPORTS_DIR / "categorical_target_rates.csv", index=False)

,feature,category,size,target_rate_pct
46,company_size,(Missing),5938,40.59
47,company_size,10/49,1471,23.39
50,company_size,10000+,2019,19.07
53,company_size,5000-9999,563,18.12
51,company_size,50-99,3083,17.68
...,...,...,...,...
19,major_discipline,Humanities,669,21.08
17,major_discipline,Arts,253,20.95
16,major_discipline,(Missing),2813,19.52
5,relevent_experience,No relevent experience,5366,33.84


In [8]:
city_cdi_check = check_city_cdi_relationship(df_raw)
display(city_cdi_check)
print(
    "Kết luận: city có 123 mức và ánh xạ 1-1 sang city_development_index; "
    "loại city giúp tránh one-hot dư thừa."
)
plot_high_cardinality_feature(
    df_raw,
    col="city",
    top_n=15,
    save_path=str(FIGURES_DIR / "group5_high_cardinality_city_top15.png"),
    show=True,
)

,Tổng số thành phố (unique cities),Số lượng CDI tối đa trên mỗi thành phố,Số lượng CDI tối thiểu trên mỗi thành phố,Quan hệ 1 - 1 (Toàn vẹn)
0,123,1,1,CÓ (100% Phù hợp)


Kết luận: city có 123 mức và ánh xạ 1-1 sang city_development_index; loại city giúp tránh one-hot dư thừa.


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group5_high_cardinality_city_top15.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:399: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
missing_target_rows = []
for col in df_raw.columns[df_raw.isna().any()]:
    grouped = df_raw.groupby(df_raw[col].isna())["target"].agg(["size", "mean"])
    missing_target_rows.append({
        "feature": col,
        "target_rate_observed_pct": grouped.loc[False, "mean"] * 100,
        "target_rate_missing_pct": grouped.loc[True, "mean"] * 100,
        "difference_pp": (grouped.loc[True, "mean"] - grouped.loc[False, "mean"]) * 100,
    })
missingness_target = pd.DataFrame(missing_target_rows).sort_values("difference_pp", ascending=False)
display(missingness_target.round(2))
missingness_target.to_csv(REPORTS_DIR / "missingness_target_rates.csv", index=False)

plot_missingness_vs_target(
    df_raw,
    save_path=str(FIGURES_DIR / "group6_missingness_vs_target.png"),
    show=True,
)

,feature,target_rate_observed_pct,target_rate_missing_pct,difference_pp
5,company_size,17.90,40.59,22.68
6,company_type,18.38,38.83,20.45
7,last_new_job,24.68,36.41,11.73
4,experience,24.90,35.38,10.49
0,gender,23.13,30.81,7.69
1,enrolled_university,24.79,31.87,7.07
2,education_level,24.99,22.61,-2.38
3,major_discipline,25.87,19.52,-6.35


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group6_missingness_vs_target.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:493: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
plot_correlation_matrix(
    df_raw,
    save_path=str(FIGURES_DIR / "group6_correlation_matrix.png"),
    show=True,
)

-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group6_correlation_matrix.png


/home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/modules/eda_viz.py:552: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Tái tạo đầy đủ các biểu đồ chi tiết từ notebook EDA trước đây.
nominal_cols = ["gender", "company_type", "major_discipline", "relevent_experience", "enrolled_university"]
for col in nominal_cols:
    plot_nominal_feature(
        df_raw,
        col,
        save_path=str(FIGURES_DIR / f"group3_nominal_{col}.png"),
        show=False,
    )

for col in ["education_level", "company_size", "experience", "last_new_job"]:
    plot_ordinal_feature(
        df_raw,
        col,
        list(ORDINAL_MAPPINGS[col].keys()),
        save_path=str(FIGURES_DIR / f"group4_ordinal_{col}.png"),
        show=False,
    )
print("Đã tái tạo đầy đủ bộ biểu đồ chi tiết trong reports/figures/.")

-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group3_nominal_gender.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group3_nominal_company_type.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group3_nominal_major_discipline.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group3_nominal_relevent_experience.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group3_nominal_enrolled_university.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group4_ordinal_education_level.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group4_ordinal_company_size.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group4_ordinal_experience.png


-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/reports/figures/group4_ordinal_last_new_job.png
Đã tái tạo đầy đủ bộ biểu đồ chi tiết trong reports/figures/.


### Kết luận EDA và quyết định kỹ thuật

- Target lệch lớp: lớp dương chiếm khoảng 25%; cần ưu tiên ROC-AUC, PR-AUC, F1/Recall thay vì chỉ Accuracy.
- `enrollee_id` chỉ là định danh; `city` dư thừa với `city_development_index`: loại cả hai khỏi mô hình.
- `company_size` và `company_type` có missingness gắn với target: giữ nhóm `Unknown`/giá trị `-1` và tạo cờ missing riêng.
- Các biến có thứ bậc được ánh xạ theo trật tự nghiệp vụ; biến danh nghĩa được one-hot.
- Feature engineering tập trung vào phi tuyến của CDI/training hours, tương tác kinh nghiệm và độ đầy đủ hồ sơ.

## 3. Chia Train / Validation / Test chống rò rỉ dữ liệu

Tỷ lệ **70% / 15% / 15%**, phân tầng theo target. Validation dùng để chọn siêu tham số hoặc early stopping; test không tham gia bất kỳ quyết định mô hình nào.

In [12]:
X_train_raw, X_validation_raw, X_test_raw, y_train, y_validation, y_test = split_train_validation_test(
    df_raw,
    validation_size=0.15,
    test_size=0.15,
    random_state=RANDOM_STATE,
    drop_unnecessary=True,
)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(y_train), len(y_validation), len(y_test)],
    "positive_count": [int(y_train.sum()), int(y_validation.sum()), int(y_test.sum())],
    "positive_rate_pct": [y_train.mean() * 100, y_validation.mean() * 100, y_test.mean() * 100],
}).set_index("split")
display(split_summary.round(3))

assert set(X_train_raw.index).isdisjoint(X_validation_raw.index)
assert set(X_train_raw.index).isdisjoint(X_test_raw.index)
assert set(X_validation_raw.index).isdisjoint(X_test_raw.index)

,rows,positive_count,positive_rate_pct
split,,,
train,13410,3343,24.929
validation,2874,717,24.948
test,2874,717,24.948


## 4. Tiền xử lý và Feature Engineering

`TabularPreprocessor` thực hiện nhất quán:

- median cho số, mode cho ordinal thông thường;
- `company_size=-1`, `company_type=Unknown` và hai cờ missing để bảo toàn tín hiệu MNAR;
- ordinal mapping và one-hot với `handle_unknown='ignore'`;
- 7 đặc trưng mới không dùng target;
- StandardScaler hoặc MinMaxScaler `[0, 1]`;
- PCA 90%/95% chỉ fit trên train.

In [13]:
engineered_feature_description = pd.DataFrame({
    "feature": ENGINEERED_FEATURES,
    "ý_nghĩa": [
        "log(1 + training_hours), giảm lệch phải",
        "quan hệ phi tuyến của mức phát triển thành phố",
        "tương tác kinh nghiệm × mức phát triển thành phố",
        "cường độ đào tạo trên một đơn vị kinh nghiệm",
        "tần suất chuyển việc tương đối theo kinh nghiệm",
        "tương tác trình độ học vấn × kinh nghiệm",
        "số trường thông tin còn thiếu trong hồ sơ",
    ],
})
display(engineered_feature_description)

,feature,ý_nghĩa
0,training_hours_log1p,"log(1 + training_hours), giảm lệch phải"
1,city_development_index_squared,quan hệ phi tuyến của mức phát triển thành phố
2,experience_x_city_development,tương tác kinh nghiệm × mức phát triển thành phố
3,training_hours_per_experience,cường độ đào tạo trên một đơn vị kinh nghiệm
4,career_mobility_ratio,tần suất chuyển việc tương đối theo kinh nghiệm
5,education_x_experience,tương tác trình độ học vấn × kinh nghiệm
6,profile_missing_count,số trường thông tin còn thiếu trong hồ sơ


In [14]:
configurations = {
    "std": TabularPreprocessor(scaling="standard", add_engineered_features=True, random_state=RANDOM_STATE),
    "minmax": TabularPreprocessor(scaling="minmax", feature_range=(0, 1), add_engineered_features=True, random_state=RANDOM_STATE),
    "pca90": TabularPreprocessor(scaling="standard", pca_variance=0.90, add_engineered_features=True, random_state=RANDOM_STATE),
    "pca95": TabularPreprocessor(scaling="standard", pca_variance=0.95, add_engineered_features=True, random_state=RANDOM_STATE),
}

transformed = {}
for name, preprocessor in configurations.items():
    transformed[f"X_train_{name}"] = preprocessor.fit_transform(X_train_raw).astype(np.float32)
    transformed[f"X_validation_{name}"] = preprocessor.transform(X_validation_raw).astype(np.float32)
    transformed[f"X_test_{name}"] = preprocessor.transform(X_test_raw).astype(np.float32)

feature_names = configurations["std"].get_feature_names_out()
shape_summary = pd.DataFrame([
    {"configuration": name, "train": transformed[f"X_train_{name}"].shape,
     "validation": transformed[f"X_validation_{name}"].shape,
     "test": transformed[f"X_test_{name}"].shape}
    for name in configurations
]).set_index("configuration")
display(shape_summary)
print(f"Tổng số đặc trưng trước PCA: {len(feature_names)}")
display(pd.DataFrame({"feature_name": feature_names}))

,train,validation,test
configuration,,,
std,"(13410, 39)","(2874, 39)","(2874, 39)"
minmax,"(13410, 39)","(2874, 39)","(2874, 39)"
pca90,"(13410, 22)","(2874, 22)","(2874, 22)"
pca95,"(13410, 24)","(2874, 24)","(2874, 24)"


Tổng số đặc trưng trước PCA: 39


,feature_name
0,city_development_index
1,training_hours
2,education_level
3,company_size
4,experience
5,last_new_job
6,company_size_isna
7,company_type_isna
8,gender_Female
9,gender_Male


In [15]:
pca_full = PCA(random_state=RANDOM_STATE).fit(transformed["X_train_std"])
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_) * 100
k90 = transformed["X_train_pca90"].shape[1]
k95 = transformed["X_train_pca95"].shape[1]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.bar(np.arange(1, len(cumulative_variance) + 1), pca_full.explained_variance_ratio_ * 100,
       alpha=0.35, label="Phương sai từng thành phần")
ax.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance,
        marker="o", markersize=3, label="Phương sai tích lũy")
for threshold, k, color in [(90, k90, "#e6550d"), (95, k95, "#d73027")]:
    ax.axhline(threshold, color=color, linestyle="--")
    ax.axvline(k, color=color, linestyle=":", label=f"{threshold}%: k={k}")
ax.set(xlabel="Số thành phần chính", ylabel="Phương sai giải thích (%)", ylim=(0, 103),
       title="PCA chỉ fit trên tập train")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "group7_pca_scree_plot.png", dpi=300, bbox_inches="tight")
plt.show()

/tmp/ipykernel_1078836/2167857505.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Xử lý mất cân bằng và gợi ý chọn representation

Notebook không resample trước khi split. Nếu dùng SMOTE/undersampling, thao tác đó phải nằm bên trong từng fold của train. `class_weight` dưới đây là lựa chọn baseline an toàn.

In [16]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
representation_guide = pd.DataFrame({
    "representation": ["std", "minmax", "pca90", "pca95"],
    "khuyến_nghị": [
        "Logistic Regression, SVM, MLP/ReLU",
        "MLP hoặc mô hình cần miền [0, 1]",
        "Mô hình tuyến tính/SVM khi ưu tiên tốc độ",
        "Cấu hình PCA giữ nhiều thông tin hơn",
    ],
})
print("Class weights:", class_weights)
display(representation_guide)

Class weights: {0: 0.6660375484255489, 1: 2.0056835177983845}


,representation,khuyến_nghị
0,std,"Logistic Regression, SVM, MLP/ReLU"
1,minmax,"MLP hoặc mô hình cần miền [0, 1]"
2,pca90,Mô hình tuyến tính/SVM khi ưu tiên tốc độ
3,pca95,Cấu hình PCA giữ nhiều thông tin hơn


## 6. Xuất artefact sẵn sàng huấn luyện

Mảng đặc trưng dùng `float32` để giảm bộ nhớ và tương thích tốt với framework Deep Learning. Preprocessor đã fit được lưu để tái sử dụng đúng cùng phép biến đổi khi suy luận.

In [17]:
feature_names_dict = {
    "full_features": feature_names,
    "engineered_features": ENGINEERED_FEATURES,
    "pca90_features": configurations["pca90"].get_feature_names_out(),
    "pca95_features": configurations["pca95"].get_feature_names_out(),
}
metadata = {
    "dataset": DATA_PATH.name,
    "total_samples": int(len(df_raw)),
    "split": {"train": int(len(y_train)), "validation": int(len(y_validation)), "test": int(len(y_test))},
    "split_ratio": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "positive_rate": {
        "train": float(y_train.mean()),
        "validation": float(y_validation.mean()),
        "test": float(y_test.mean()),
    },
    "random_state": RANDOM_STATE,
    "dropped_columns": ["enrollee_id", "city"],
    "class_weights": class_weights,
    "configurations": {
        "full_features": int(len(feature_names)),
        "pca_90_components": int(k90),
        "pca_95_components": int(k95),
        "dtype": "float32",
    },
    "versions": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scikit_learn": sklearn.__version__},
}

save_processed_features(
    output_dir=str(FEATURES_DIR),
    feature_sets=transformed,
    y_train=y_train.to_numpy(dtype=np.int64),
    y_validation=y_validation.to_numpy(dtype=np.int64),
    y_test=y_test.to_numpy(dtype=np.int64),
    feature_names_dict=feature_names_dict,
    metadata=metadata,
)
joblib.dump(configurations, FEATURES_DIR / "preprocessors.joblib")
np.savez_compressed(
    FEATURES_DIR / "split_indices.npz",
    train=X_train_raw.index.to_numpy(),
    validation=X_validation_raw.index.to_numpy(),
    test=X_test_raw.index.to_numpy(),
)
print(f"Đã xuất artefact vào: {FEATURES_DIR}")

-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_train_std.npy [shape: (13410, 39)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_validation_std.npy [shape: (2874, 39)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_test_std.npy [shape: (2874, 39)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_train_minmax.npy [shape: (13410, 39)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_validation_minmax.npy [shape: (2874, 39)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_test_minmax.npy [shape: (2874, 39)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_train_pca90.npy [shape: (13410, 22)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_validation_pca90.npy [shape: (2874, 22)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignment-G3/features/X_test_pca90.npy [shape: (2874, 22)]
-> Đã lưu: /home/puffy/ML/HCMUT/BTL/261-ML-Assignmen

In [18]:
loaded = load_processed_features(str(FEATURES_DIR))
checks = []
expected_rows = {"train": len(y_train), "validation": len(y_validation), "test": len(y_test)}
for name, arr in sorted(loaded.items()):
    split = next((s for s in expected_rows if f"_{s}_" in f"_{name}_"), None)
    row_ok = split is None or arr.shape[0] == expected_rows[split]
    checks.append({
        "name": name,
        "shape": str(arr.shape),
        "dtype": str(arr.dtype),
        "finite": bool(np.isfinite(arr).all()),
        "row_count_ok": bool(row_ok),
    })
integrity = pd.DataFrame(checks)
display(integrity)
assert integrity["finite"].all() and integrity["row_count_ok"].all()
assert len(configurations["std"].get_feature_names_out()) == loaded["X_train_std"].shape[1]
print("KIỂM ĐỊNH THÀNH CÔNG: không NaN/Inf, số dòng và số đặc trưng nhất quán.")

,name,shape,dtype,finite,row_count_ok
0,X_test_minmax,"(2874, 39)",float32,True,True
1,X_test_pca90,"(2874, 22)",float32,True,True
2,X_test_pca95,"(2874, 24)",float32,True,True
3,X_test_std,"(2874, 39)",float32,True,True
4,X_train_minmax,"(13410, 39)",float32,True,True
5,X_train_pca90,"(13410, 22)",float32,True,True
6,X_train_pca95,"(13410, 24)",float32,True,True
7,X_train_std,"(13410, 39)",float32,True,True
8,X_validation_minmax,"(2874, 39)",float32,True,True
9,X_validation_pca90,"(2874, 22)",float32,True,True


KIỂM ĐỊNH THÀNH CÔNG: không NaN/Inf, số dòng và số đặc trưng nhất quán.


## 7. Smoke test: xác nhận đầu ra huấn luyện được

Đây không phải tuning hay kết quả cuối. Hai mô hình nhỏ chỉ dùng **validation**, không đụng tới test, nhằm xác nhận data contract cho cả ML và neural network.

In [19]:
smoke_models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "MLPClassifier": MLPClassifier(
        hidden_layer_sizes=(64, 32), batch_size=128, max_iter=60,
        early_stopping=True, validation_fraction=0.15,
        n_iter_no_change=8, random_state=RANDOM_STATE,
    ),
}
smoke_rows = []
for name, model in smoke_models.items():
    model.fit(transformed["X_train_std"], y_train)
    validation_probability = model.predict_proba(transformed["X_validation_std"])[:, 1]
    validation_prediction = (validation_probability >= 0.5).astype(int)
    smoke_rows.append({
        "model": name,
        "validation_roc_auc": roc_auc_score(y_validation, validation_probability),
        "validation_f1": f1_score(y_validation, validation_prediction),
        "validation_recall": recall_score(y_validation, validation_prediction),
    })
smoke_results = pd.DataFrame(smoke_rows).set_index("model")
display(smoke_results.round(4))

,validation_roc_auc,validation_f1,validation_recall
model,,,
LogisticRegression,0.7815,0.5816,0.7406
MLPClassifier,0.7774,0.4250,0.3417


## 8. Mẫu nạp dữ liệu cho notebook huấn luyện tiếp theo

```python
import numpy as np

X_train = np.load("features/X_train_std.npy")
X_validation = np.load("features/X_validation_std.npy")
X_test = np.load("features/X_test_std.npy")
y_train = np.load("features/y_train.npy")
y_validation = np.load("features/y_validation.npy")
y_test = np.load("features/y_test.npy")
```

Với Keras/PyTorch, dùng `X_*_std.npy` hoặc `X_*_minmax.npy`, lấy `input_dim = X_train.shape[1]`, dùng validation cho early stopping và chỉ báo cáo test sau khi đã khóa kiến trúc/siêu tham số.

## Kết quả bàn giao

- EDA có kiểm tra chất lượng, mất cân bằng, missingness/MNAR, quan hệ biến số/phân loại và quyết định loại cột.
- Pipeline chống leakage, hỗ trợ giá trị chưa thấy khi inference.
- 7 đặc trưng mới dựa trên logic nghiệp vụ, không sử dụng target.
- 4 representation (`std`, `minmax`, `pca90`, `pca95`) cho ba split.
- Artefact và metadata đã được kiểm định, sẵn sàng cho ML và Deep Learning.